# Cornell Tech Human vs. AI — Classical ML Experiments 

**Goal:** improve hidden-test AUROC by comparing algorithms, features, regularization, and validation schemes, using **only the provided CSV data**. No pretrained transformers, external datasets, or LLM APIs.


**Workflow:** load data → train/validation split → candidate algorithms → cross-validation of finalists → optional distribution-shift diagnostic → refit and submit. Keep a record of experiment results for your course report.

## 1. Setup

In VS Code, select the local Python environment you want to use as the notebook kernel, then run the dependency cell below. The notebook reads the competition CSV files from the repository's `data/raw` directory.

In [ ]:
%pip install numpy pandas matplotlib scipy scikit-learn

In [ ]:
import os, re, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.validation import check_is_fitted
from scipy.sparse import issparse
from scipy.special import expit

SEED = 42
np.random.seed(SEED)
print('Scikit-learn version:', __import__('sklearn').__version__)
search_roots = (Path.cwd(), *Path.cwd().parents)
try:
    REPO_ROOT = next(root for root in search_roots if (root / 'data' / 'raw').is_dir())
except StopIteration as exc:
    raise FileNotFoundError('Could not find the repository data/raw directory from the current working directory.') from exc
DATA_DIR = REPO_ROOT / 'data' / 'raw'
required_data_files = ('train.csv', 'test.csv')
missing_data_files = [name for name in required_data_files if not (DATA_DIR / name).is_file()]
if missing_data_files:
    raise FileNotFoundError(f'Missing required files in {DATA_DIR}: {missing_data_files}')
print('Data directory:', DATA_DIR)

## 2. Load and inspect the labeled data

`train_starter_2000.csv` is a subset of `train.csv` and should **not** be added as extra training rows.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
assert {'id','text','label'}.issubset(train.columns)
assert {'id','text'}.issubset(test.columns)
assert train['id'].is_unique and test['id'].is_unique
train['text'] = train['text'].fillna('').astype(str)
test['text'] = test['text'].fillna('').astype(str)
train['label'] = train['label'].astype(int)
assert set(train.label.unique()) <= {0,1}
conflict = train.groupby('text')['label'].nunique()
bad_texts = set(conflict[conflict > 1].index)
data = train.loc[~train['text'].isin(bad_texts)].drop_duplicates('text').reset_index(drop=True)
print('Train, usable, test:', len(train), len(data), len(test))
print('Label counts:', data.label.value_counts().to_dict())
print('Contradictory duplicate texts removed:', len(bad_texts))
print('Exact test/train text overlaps:', len(set(test.text) & set(data.text)))
print('Text-length summary by class:')
display(data.assign(words=data.text.str.split().str.len(), chars=data.text.str.len()).groupby('label')[['words','chars']].agg(['mean','median','std']).round(1))

## 3. Fixed holdout split

All candidate models use the **same untouched validation set**. After selecting finalists, use multiple folds to check whether the ranking is stable. This split is still random, so it may not capture hidden-test domain changes.

In [ ]:
Xtr, Xva, ytr, yva = train_test_split(data.text, data.label, test_size=0.20, stratify=data.label, random_state=SEED)
print('Train:',len(Xtr),'Validation:',len(Xva))

## 4. Feature representations and algorithms

**Feature engineering**: word uni/bi-grams; character n-grams; combined TF-IDF; explicit writing-style features. **Algorithms**: logistic regression (L2 regularization), calibrated linear SVM, Complement Naive Bayes, and gradient-boosted trees. All learn from scratch.

We limit vocab sizes to keep Colab CPU manageable. Adjust `max_features` only after making a first comparison.

In [ ]:
def word_vec():
    return TfidfVectorizer(analyzer='word', ngram_range=(1,2), min_df=2, max_df=.98,
                           max_features=100_000, sublinear_tf=True, strip_accents='unicode', dtype=np.float32)

def char_vec():
    return TfidfVectorizer(analyzer='char', ngram_range=(3,5), min_df=2,
                           max_features=180_000, sublinear_tf=True, dtype=np.float32)

def combined_vec():
    return FeatureUnion([('word', word_vec()), ('char', char_vec())])

# SVM decision scores are good for AUROC; calibration gives probability values for Kaggle.
def calibrated_svm(C=1.0):
    return CalibratedClassifierCV(LinearSVC(C=C, random_state=SEED, max_iter=4000),
                                   method='sigmoid', cv=3)

CANDIDATES = {
    'char_logreg_C2': Pipeline([('features',char_vec()),('clf',LogisticRegression(C=2, max_iter=1500))]),
    'char_logreg_C8': Pipeline([('features',char_vec()),('clf',LogisticRegression(C=8, max_iter=1500))]),
    'combined_logreg_C2': Pipeline([('features',combined_vec()),('clf',LogisticRegression(C=2, max_iter=1500))]),
    'combined_logreg_C8': Pipeline([('features',combined_vec()),('clf',LogisticRegression(C=8, max_iter=1500))]),
    'char_svm_C1': Pipeline([('features',char_vec()),('clf',calibrated_svm(1))]),
    'combined_svm_C1': Pipeline([('features',combined_vec()),('clf',calibrated_svm(1))]),
    'word_complementNB': Pipeline([('features',word_vec()),('clf',ComplementNB(alpha=0.5))]),
}
print('Candidate models:',list(CANDIDATES))

## 5. Compare candidates on one holdout

Only the **validation** labels are used to compare models. AUROC measures ranking, not classification accuracy. Avoid Kaggle submissions for routine hyperparameter selection.

In [ ]:
results = []
trained = {}
val_predictions = {}
for name, estimator in CANDIDATES.items():
    t0=time.time()
    model=clone(estimator)
    model.fit(Xtr, ytr)
    prob=model.predict_proba(Xva)[:,1]
    score=roc_auc_score(yva,prob)
    results.append({'name':name,'holdout_auc':score,'seconds':round(time.time()-t0,1)})
    trained[name]=model
    val_predictions[name]=prob
    print(f'{name:24} AUROC={score:.5f}  elapsed={results[-1]["seconds"]}s')
results_df=pd.DataFrame(results).sort_values('holdout_auc',ascending=False).reset_index(drop=True)
display(results_df)

### 5.5 Experiment: Sample Size Scaling (Learning Curve)
We evaluate the selected model configuration under different training sample sizes (500, 1,000, 2,000, 5,000, and 8,000) drawn from the 8,000-row training split (`Xtr`, `ytr`), keeping the validation set (`Xva`, `yva`) constant.

In [ ]:
sample_sizes = [500, 1000, 2000, 5000, 8000]
scaling_results = []

# Using the highest performing text model configuration from candidate selection
scaling_model_template = CANDIDATES['combined_svm_C1']

for size in sample_sizes:
    # Sample exactly 'size' instances from Xtr/ytr
    Xtr_sub = Xtr.iloc[:size]
    ytr_sub = ytr.iloc[:size]

    # Clone and train
    model = clone(scaling_model_template)
    t0 = time.time()
    model.fit(Xtr_sub, ytr_sub)
    elapsed = time.time() - t0

    # Predict and evaluate on both the training subset and the validation set
    train_prob = model.predict_proba(Xtr_sub)[:, 1]
    train_score = roc_auc_score(ytr_sub, train_prob)

    val_prob = model.predict_proba(Xva)[:, 1]
    val_score = roc_auc_score(yva, val_prob)

    scaling_results.append({
        'sample_size': size,
        'train_auroc': train_score,
        'validation_auroc': val_score,
        'training_time_seconds': round(elapsed, 2)
    })
    print(f"Size: {size:5d} | Train AUROC: {train_score:.5f} | Val AUROC: {val_score:.5f} | Time: {elapsed:.2f}s")

# Convert to DataFrame and display
scaling_df = pd.DataFrame(scaling_results)
display(scaling_df)

# Plot the learning curve containing both training and validation curves
plt.figure(figsize=(10, 6))
plt.plot(scaling_df['sample_size'], scaling_df['train_auroc'], marker='o', linestyle='--', color='r', linewidth=2, label='Training AUROC')
plt.plot(scaling_df['sample_size'], scaling_df['validation_auroc'], marker='o', linestyle='-', color='b', linewidth=2, label='Validation AUROC')
plt.title('Learning Curve: AUROC vs. Training Sample Size')
plt.xlabel('Training Sample Size')
plt.ylabel('AUROC')
plt.legend(loc='best')
plt.grid(True, linestyle='--', alpha=0.6)
plt.ylim(0.8, 1.01)
plt.show()

### 📊 Subgroup Analysis: Performance Across Text-Length Bins

To ensure our model is robust and does not rely on text length as a shortcut, we split our validation set into three equal-sized bins based on word count:
* **Short Texts**: $[0, 33^{rd}\text{ percentile}]$ of words
* **Medium Texts**: $(33^{rd}, 66^{th}\text{ percentile}]$ of words
* **Long Texts**: $(66^{th}\text{ percentile}, \infty)$ of words

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# 1. Prepare validation DataFrame with word counts
va_df = pd.DataFrame({
    'text': Xva,
    'label': yva,
    'pred_prob': val_predictions['combined_svm_C1']
})
va_df['word_count'] = va_df['text'].apply(lambda x: len(x.split()))

# 2. Define text-length bins using quantiles
q33 = va_df['word_count'].quantile(0.33)
q66 = va_df['word_count'].quantile(0.66)

def get_bin(wc):
    if wc <= q33:
        return f'Short (<= {int(q33)} words)'
    elif wc <= q66:
        return f'Medium ({int(q33)} - {int(q66)} words)'
    else:
        return f'Long (> {int(q66)} words)'

va_df['length_bin'] = va_df['word_count'].apply(get_bin)

# 3. Calculate AUROC and metrics for each length bin
subgroup_analysis = []
bins_order = [f'Short (<= {int(q33)} words)', f'Medium ({int(q33)} - {int(q66)} words)', f'Long (> {int(q66)} words)']

for b in bins_order:
    sub_df = va_df[va_df['length_bin'] == b]
    y_true = sub_df['label']
    y_pred = sub_df['pred_prob']

    # Check if we have both classes represented in the subgroup
    if len(np.unique(y_true)) > 1:
        auc = roc_auc_score(y_true, y_pred)
    else:
        auc = np.nan

    subgroup_analysis.append({
        'Text Length Group': b,
        'Count': len(sub_df),
        'Human Count (0)': (y_true == 0).sum(),
        'AI Count (1)': (y_true == 1).sum(),
        'Subgroup AUROC': round(auc, 5)
    })

subgroup_df = pd.DataFrame(subgroup_analysis)
display(subgroup_df)

### 🔍 Error Analysis: Diagnostic Sample of False Positives and False Negatives

To understand where the `combined_svm_C1` model struggles, we run diagnostic code to extract 5 False Positives ($y_{true} = 0$, $P(AI) \ge 0.5$) and 5 False Negatives ($y_{true} = 1$, $P(AI) < 0.5$).

In [ ]:
# Prepare analysis DataFrame
err_df = pd.DataFrame({
    'text_idx': Xva.index,  # Original index from the dataset
    'text': Xva.values,
    'label': yva.values,
    'pred_prob': val_predictions['combined_svm_C1']
})

# Classify predictions using 0.5 threshold
err_df['pred_label'] = (err_df['pred_prob'] >= 0.5).astype(int)

# Filter for False Positives (Human predicted as AI)
fps = err_df[(err_df['label'] == 0) & (err_df['pred_label'] == 1)].sort_values(by='pred_prob', ascending=False)

# Filter for False Negatives (AI predicted as Human)
fns = err_df[(err_df['label'] == 1) & (err_df['pred_label'] == 0)].sort_values(by='pred_prob', ascending=True)

print("=== TOP FALSE POSITIVES ===")
for idx, row in fps.head(5).iterrows():
    print(f"ID/Index: {row['text_idx']} | P(AI): {row['pred_prob']:.4f}")
    print(f"Snippet: {row['text'][:250]}...\n")

print("=== TOP FALSE NEGATIVES ===")
for idx, row in fns.head(5).iterrows():
    print(f"ID/Index: {row['text_idx']} | P(AI): {row['pred_prob']:.4f}")
    print(f"Snippet: {row['text'][:250]}...\n")

## 6. Experiment: handcrafted stylometric features

This model ignores the TF-IDF vocabulary and instead learns from measurable features like length, lexical diversity, punctuation, sentence-length variance, and formatting. It may generalize differently from token models. If a feature reflects a dataset artifact (for example, text length), it may fail on the hidden test set, so treat this as an experiment, not a guarantee.

In [ ]:
PUNCT = [',','.',';',':','!','?','Item_1',"'",'(',')','-','\n']
COMMON = ['the','and','of','to','in','is','that','for','with','as','on','it','this','however','moreover','therefore','i','we','you']
def style_features(texts):
    rows=[]
    for t in texts:
        words=re.findall(r"\b[\w']+\b",t.lower())
        n=max(len(words),1); nc=max(len(t),1)
        sentences=[x for x in re.split(r'[.!?]+',t) if x.strip()]
        lens=[len(re.findall(r"\b[\w']+\b",s)) for s in sentences] or [0]
        counts={w:words.count(w) for w in COMMON}
        feats=[np.log1p(len(t)),np.log1p(len(words)),np.mean([len(w) for w in words]) if words else 0,
               len(set(words))/n, np.mean(lens),np.std(lens),
               sum(c.isupper() for c in t)/nc,sum(c.isdigit() for c in t)/nc,
               sum(c.isspace() for c in t)/nc,
               len(sentences)/n, sum(len(w)>=8 for w in words)/n]
        feats += [t.count(p)/nc for p in PUNCT]
        feats += [counts[w]/n for w in COMMON]
        rows.append(feats)
    return np.asarray(rows,dtype=np.float32)

style_tr = style_features(Xtr)
style_va = style_features(Xva)
style_model = HistGradientBoostingClassifier(max_iter=180, learning_rate=0.05,
                                             max_leaf_nodes=15, l2_regularization=2,
                                             random_state=SEED)
style_model.fit(style_tr,ytr)
style_prob=style_model.predict_proba(style_va)[:,1]
style_score=roc_auc_score(yva,style_prob)
val_predictions['style_boosting']=style_prob
print('Stylometric boosting AUROC:',round(style_score,5))

## 7. Compare simple validation ensembles

An ensemble can improve robustness **only if its errors are complementary**. We test fixed combinations on the holdout and never blend in any hidden Kaggle labels. Ensemble weight tuning on one split can overfit, so use simple weights and then validate the finalists across multiple folds.

In [ ]:
# Blend a strong text model with the stylistic model. Fixed blend weights avoid excessive tuning.
best_text_name=results_df.iloc[0]['name']
base=val_predictions[best_text_name]
blend_rows=[]
for name,pred in [('best_text_only',base), ('text_90_style_10',0.9*base+0.1*style_prob),
                  ('text_75_style_25',0.75*base+0.25*style_prob)]:
    blend_rows.append((name,roc_auc_score(yva,pred)))
display(pd.DataFrame(blend_rows,columns=['approach','holdout_auc']).sort_values('holdout_auc',ascending=False))

## 8. Cross-validate the two leading text models

Cross-validation provides a more stable estimate than a single split. **If performance is high across random folds but much worse on Kaggle, optimizing another random split alone is unlikely to fix the generalization gap.** This may take several minutes on CPU, especially for calibrated SVMs.

In [ ]:
RUN_CV = True
CV_FOLDS = 3
cv_results=[]
if RUN_CV:
    top_models=results_df.head(2)['name'].tolist()
    splitter=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=123)
    for name in top_models:
        t0=time.time()
        scores=cross_val_score(clone(CANDIDATES[name]),data.text,data.label,cv=splitter,
                               scoring='roc_auc',n_jobs=1)
        cv_results.append({'name':name,'cv_mean_auc':scores.mean(),'cv_std':scores.std(),
                           'fold_auc':np.round(scores,5).tolist()})
        print(name,'fold AUCs',np.round(scores,5),'mean',round(scores.mean(),5),
              'elapsed seconds',round(time.time()-t0,1))
    display(pd.DataFrame(cv_results).sort_values('cv_mean_auc',ascending=False))
else:
    print('CV skipped; choose using holdout results.')

## 9. Optional diagnostic: can a classifier distinguish TRAIN texts from TEST texts?

This is *not* a label prediction model. It tests whether the text distributions differ. AUC near 0.5 means the diagnostic cannot distinguish them; a higher AUC suggests a measurable difference. **High diagnostic AUC does not prove** that domain shift caused the Kaggle gap, but helps prioritize robust feature design. Do not use test labels (none are provided).

In [ ]:
RUN_SHIFT_DIAGNOSTIC = True
if RUN_SHIFT_DIAGNOSTIC:
    shift_X=pd.concat([data.text.rename('text'), test.text.rename('text')],ignore_index=True)
    shift_y=np.r_[np.zeros(len(data),dtype=int),np.ones(len(test),dtype=int)]
    sx_tr,sx_va,sy_tr,sy_va=train_test_split(shift_X,shift_y,stratify=shift_y,test_size=.25,random_state=17)
    shift_model=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),max_features=30000,min_df=3,
                                                  sublinear_tf=True)),
                          ('clf',LogisticRegression(C=1,max_iter=1000))])
    shift_model.fit(sx_tr,sy_tr)
    shift_auc=roc_auc_score(sy_va,shift_model.predict_proba(sx_va)[:,1])
    print('Train-versus-test distinguishability AUROC:',round(shift_auc,5))
    print('Interpret cautiously: this tests text distribution, not AI authorship.')

### 9.5 Semi-Supervised Learning: Pseudo-Labeling
We leverage `unlabeled.csv` to capture the underlying document distribution of the task. By running our top-performing text classifier on the unlabeled texts, picking the highest-confidence predictions, and adding them back to our training pool with temporary 'pseudo-labels', we help the decision boundary generalize better to the unseen domain.

In [ ]:
# 1. Load unlabeled data
unlabeled_path = DATA_DIR / 'unlabeled.csv'
if unlabeled_path.exists():
    unlabeled = pd.read_csv(unlabeled_path)
    unlabeled['text'] = unlabeled['text'].fillna('').astype(str)

    # 2. Fit our selected text model on the original full training set
    print("Fitting initial teacher model on original training data...")
    teacher_model = clone(CANDIDATES['combined_svm_C1'])
    teacher_model.fit(data.text, data.label)

    # 3. Predict probabilities on unlabeled data
    unlabeled_probs = teacher_model.predict_proba(unlabeled.text)[:, 1]

    # 4. Set strict confidence thresholds to select pseudo-labels
    CONFIDENCE_THRESHOLD_HIGH = 0.98
    CONFIDENCE_THRESHOLD_LOW = 0.02

    pseudo_pos_mask = unlabeled_probs > CONFIDENCE_THRESHOLD_HIGH
    pseudo_neg_mask = unlabeled_probs < CONFIDENCE_THRESHOLD_LOW

    pseudo_pos = unlabeled[pseudo_pos_mask].copy()
    pseudo_pos['label'] = 1

    pseudo_neg = unlabeled[pseudo_neg_mask].copy()
    pseudo_neg['label'] = 0

    pseudo_labeled_df = pd.concat([pseudo_pos, pseudo_neg], ignore_index=True)
    print(f"Generated {len(pseudo_pos)} positive pseudo-labels (p > {CONFIDENCE_THRESHOLD_HIGH})")
    print(f"Generated {len(pseudo_neg)} negative pseudo-labels (p < {CONFIDENCE_THRESHOLD_LOW})")
    print(f"Total pseudo-labeled training instances added: {len(pseudo_labeled_df)}")

    # 5. Concatenate original training data with the new pseudo-labeled subset
    augmented_data = pd.concat([data[['text', 'label']], pseudo_labeled_df[['text', 'label']]], ignore_index=True)
    print(f"New augmented training set size: {len(augmented_data)}")
else:
    print("unlabeled.csv not found; skipping pseudo-labeling setup.")
    augmented_data = data.copy()

We will now adapt the final select and refit step to use our augmented dataset.

In [ ]:
# Use the newly augmented dataset to train the final submission model
FINAL_MODEL_NAME = 'combined_svm_C1'

assert FINAL_MODEL_NAME in CANDIDATES
final_model = clone(CANDIDATES[FINAL_MODEL_NAME])

# Train on augmented dataset containing both human-verified and pseudo-labeled instances
final_model.fit(augmented_data.text, augmented_data.label)
print('Final model retrained on augmented dataset:', FINAL_MODEL_NAME)


## 10. Select and refit the final model

**Recommended:** select by cross-validation mean AUROC if enabled, not Kaggle public score. Try `FINAL_MODEL_NAME='combined_svm_C1'` or another candidate **only after** comparing it fairly. Pseudo-labeling is excluded from the first round because low-confidence labels can amplify distribution-shift errors.

For a blend with stylometry, explicitly set `USE_STYLE_BLEND=True` and validate that choice in Section 7.

In [ ]:
if cv_results:
    FINAL_MODEL_NAME=max(cv_results,key=lambda x:x['cv_mean_auc'])['name']
else:
    FINAL_MODEL_NAME=best_text_name

# You may override after examining the table, for example:
# FINAL_MODEL_NAME = 'combined_svm_C1'
USE_STYLE_BLEND = False
STYLE_BLEND_WEIGHT = 0.10

assert FINAL_MODEL_NAME in CANDIDATES
final_model=clone(CANDIDATES[FINAL_MODEL_NAME])
final_model.fit(data.text,data.label)
print('Final model:',FINAL_MODEL_NAME)
if USE_STYLE_BLEND:
    style_final=clone(style_model)
    style_final.fit(style_features(data.text),data.label)
    print('Blending stylometric probabilities, weight=',STYLE_BLEND_WEIGHT)

## 11. Generate and verify `submission.csv`

Outputs the required **probability of AI authorship**, never binary labels. The test ID order is preserved.

In [ ]:
probs = final_model.predict_proba(test.text)[:, 1]
if USE_STYLE_BLEND:
    probs = (1 - STYLE_BLEND_WEIGHT) * probs + STYLE_BLEND_WEIGHT * style_final.predict_proba(style_features(test.text))[:, 1]

submission = pd.DataFrame({'id': test.id, 'label': probs})
assert submission.columns.tolist() == ['id', 'label']
assert len(submission) == len(test) == 3000
assert submission.id.is_unique
assert submission.id.tolist() == test.id.tolist()
assert np.isfinite(probs).all() and ((probs >= 0) & (probs <= 1)).all()

# Standard submission path for competition upload
standard_filename = DATA_DIR / 'submission.csv'
submission.to_csv(standard_filename, index=False)
print('Saved standard submission:', standard_filename)

# Create a descriptive name using model configuration and validation score
cv_score_str = f"_cv{cv_results[0]['cv_mean_auc']:.5f}".replace('.', '') if cv_results else ''
style_suffix = f"_style_blend_{STYLE_BLEND_WEIGHT}" if USE_STYLE_BLEND else ''
descriptive_name = f"submission_{FINAL_MODEL_NAME}_pseudo_labeled{cv_score_str}{style_suffix}.csv"
descriptive_filename = DATA_DIR / descriptive_name
submission.to_csv(descriptive_filename, index=False)
print('Saved descriptive submission:', descriptive_filename)

print(submission.label.describe())
display(submission.head())

try:
    from google.colab import files
    # Download both files for convenience
    files.download(str(standard_filename))
    files.download(str(descriptive_filename))
except ImportError:
    pass

## 12. Experiment log and possible next steps

For the report, document (1) feature representation, (2) algorithm, (3) regularization, (4) holdout AUROC, (5) CV mean/std, (6) Kaggle score, and (7) generalization explanation. Keep a holdout untouched while making substantive modeling decisions; multiple tries on the same holdout can overfit to it.

Suggested further *classical ML* experiments (one change at a time):

- **Linear SVM:** try `C=0.25, 0.5, 2, 4` and compare 3-fold AUROC.
- **Character n-grams:** test `(2,5)` or `(3,6)` and higher `max_features`.
- **Word features:** investigate `min_df=1` or uni/tri-grams if CPU memory allows.
- **Regularization:** compare logistic regression `C=0.5, 2, 8` and optional elastic net with `solver='saga'`.
- **Shift robustness:** test by partitioning length/topic groups into holdouts, and look at errors by length bin.
- **Semi-supervised learning:** only after a reliable baseline, try high-confidence pseudo-labels from `unlabeled.csv` and assess gains on an untouched validation fold.

**Limitations:** without seeing the actual text rows or the competition's training/test generation processes, we cannot confirm why 0.86706 was obtained or guarantee 0.93+. The notebook tests hypotheses rather than relying on unverified external models.